# SEDS 537 Take-Home Midterm

This notebook contains the full experimental workflow for all five questions. It is the main executable submission artifact.

## Setup

Run the next cell first. It installs any missing packages into the active Jupyter kernel before the main imports run.


In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
requirements_path = ROOT / "requirements.txt"
required_modules = [
    "pandas",
    "sklearn",
    "matplotlib",
    "seaborn",
    "imblearn",
    "xgboost",
    "torch",
    "torchvision",
]

def import_status(module_name):
    try:
        __import__(module_name)
        return True, None
    except Exception as exc:
        return False, exc

print("Python:", sys.executable)
failed = []
for module_name in required_modules:
    ok, error = import_status(module_name)
    if ok:
        print(f"{module_name}: OK")
    else:
        print(f"{module_name}: not ready -> {error}")
        failed.append(module_name)

if failed:
    conda_bin = shutil.which("conda")
    if "xgboost" in failed and conda_bin:
        print("Repairing xgboost through conda...")
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "xgboost"])
        subprocess.check_call([conda_bin, "install", "-y", "xgboost"])

    remaining = []
    for module_name in required_modules:
        ok, error = import_status(module_name)
        if not ok:
            remaining.append(module_name)

    if remaining:
        print("Installing remaining packages with pip:", ", ".join(remaining))
        commands = [
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
            [sys.executable, "-m", "pip", "install", "--user", "-r", str(requirements_path)],
        ]
        last_error = None
        for command in commands:
            try:
                print("Running:", " ".join(command))
                subprocess.check_call(command)
                break
            except subprocess.CalledProcessError as exc:
                last_error = exc
                print("Install attempt failed.")
        else:
            raise RuntimeError("Automatic package installation failed.") from last_error

    print("Setup finished. Restart the kernel once, then rerun the notebook from the top.")
else:
    print("All required packages are already installed and importable.")


Python: /opt/anaconda3/bin/python
pandas: OK
sklearn: OK
matplotlib: OK
seaborn: OK
imblearn: OK
xgboost: OK
torch: OK
torchvision: OK
All required packages are already installed and importable.


In [2]:
from __future__ import annotations

import inspect
import json
import math
import os
import random
import shutil
import textwrap
import warnings
from copy import deepcopy
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.manifold import TSNE
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
REPORT_DIR = ROOT / "report"


def seed_everything(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dirs() -> None:
    for path in [DATA_DIR, OUTPUT_DIR, REPORT_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def question_dir(name: str) -> Path:
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_dataframe(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    df.to_csv(path, index=index)


def write_json(payload: dict, path: Path) -> None:
    path.write_text(json.dumps(payload, indent=2))


def metric_float(value: float | int) -> float:
    return float(value) if value is not None and not math.isnan(value) else float("nan")


def safe_percentage(series: pd.Series) -> pd.Series:
    return (series / series.sum()) * 100


def relative_to_report(path: Path) -> str:
    return path.relative_to(ROOT).as_posix()


def save_plot(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


def select_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def run_question1() -> dict:
    out_dir = question_dir("question1")
    dataset = fetch_california_housing(as_frame=True)
    df = dataset.frame.copy()
    X = df.drop(columns=["MedHouseVal"])
    y = df["MedHouseVal"]

    fig, axes = plt.subplots(3, 3, figsize=(14, 10))
    for ax, col in zip(axes.flatten(), df.columns):
        ax.hist(df[col], bins=30, color="#2563eb", alpha=0.85)
        ax.set_title(col)
    hist_path = out_dir / "histograms.png"
    save_plot(hist_path)

    corr = df.corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
    plt.title("California Housing Correlation Matrix")
    corr_path = out_dir / "correlation_matrix.png"
    save_plot(corr_path)

    scatter_features = ["MedInc", "AveRooms", "Latitude"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, feature in zip(axes, scatter_features):
        ax.scatter(df[feature], y, s=8, alpha=0.25, color="#0f766e")
        ax.set_xlabel(feature)
        ax.set_ylabel("MedHouseVal")
        ax.set_title(f"{feature} vs MedHouseVal")
    scatter_path = out_dir / "scatter_plots.png"
    save_plot(scatter_path)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    models = {
        "Linear Regression": Pipeline(
            [("identity", "passthrough"), ("model", LinearRegression())]
        ),
        "Ridge Regression": Pipeline(
            [
                ("identity", "passthrough"),
                ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
            ]
        ),
        "Random Forest": Pipeline(
            [
                ("identity", "passthrough"),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=200,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
    }

    def evaluate_regression(experiment_name: str, estimators: dict) -> tuple[pd.DataFrame, dict]:
        rows = []
        predictions = {}
        for model_name, estimator in estimators.items():
            estimator.fit(X_train, y_train)
            preds = estimator.predict(X_test)
            rows.append(
                {
                    "experiment": experiment_name,
                    "model": model_name,
                    "RMSE": metric_float(np.sqrt(mean_squared_error(y_test, preds))),
                    "MAE": metric_float(mean_absolute_error(y_test, preds)),
                    "R2": metric_float(r2_score(y_test, preds)),
                }
            )
            predictions[model_name] = preds
        return pd.DataFrame(rows), predictions

    baseline_results, baseline_preds = evaluate_regression("baseline", deepcopy(models))

    scaled_models = {
        "Linear Regression": Pipeline(
            [("scaler", StandardScaler()), ("model", LinearRegression())]
        ),
        "Ridge Regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
            ]
        ),
        "Random Forest": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=200,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
    }
    scaled_results, _ = evaluate_regression("scaled", scaled_models)

    poly_models = {
        "Linear Regression": Pipeline(
            [
                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ("scaler", StandardScaler()),
                ("model", LinearRegression()),
            ]
        ),
        "Ridge Regression": Pipeline(
            [
                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ("scaler", StandardScaler()),
                ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
            ]
        ),
        "Random Forest": Pipeline(
            [
                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=200,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
    }
    poly_results, _ = evaluate_regression("polynomial_degree_2", poly_models)

    combined_results = pd.concat([baseline_results, scaled_results, poly_results], ignore_index=True)
    save_dataframe(combined_results, out_dir / "regression_metrics.csv")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, model_name in zip(axes, baseline_preds):
        residuals = y_test - baseline_preds[model_name]
        ax.scatter(baseline_preds[model_name], residuals, s=8, alpha=0.3, color="#b91c1c")
        ax.axhline(0, color="black", linestyle="--", linewidth=1)
        ax.set_title(model_name)
        ax.set_xlabel("Fitted")
        ax.set_ylabel("Residual")
    residual_path = out_dir / "residual_plots.png"
    save_plot(residual_path)

    best_baseline = baseline_results.sort_values("RMSE").iloc[0]
    summary = {
        "dataset_rows": int(df.shape[0]),
        "baseline_best_model": best_baseline["model"],
        "baseline_best_rmse": metric_float(best_baseline["RMSE"]),
        "baseline_best_r2": metric_float(best_baseline["R2"]),
        "figures": {
            "histograms": relative_to_report(hist_path),
            "correlation_matrix": relative_to_report(corr_path),
            "scatter_plots": relative_to_report(scatter_path),
            "residual_plots": relative_to_report(residual_path),
        },
        "metrics_file": relative_to_report(out_dir / "regression_metrics.csv"),
    }
    return {
        "summary": summary,
        "table": combined_results,
    }


def build_q2_models() -> dict:
    return {
        "Logistic Regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000, n_jobs=-1)),
            ]
        ),
        "Random Forest": Pipeline(
            [
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=100,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                )
            ]
        ),
        "XGBoost": Pipeline(
            [
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="aucpr",
                        n_estimators=40,
                        max_depth=4,
                        learning_rate=0.08,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        tree_method="hist",
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                )
            ]
        ),
    }


def run_question2() -> dict:
    out_dir = question_dir("question2")
    dataset = fetch_openml(name="creditcard", version=1, as_frame=True)
    X_full = dataset.data.copy()
    y_full = dataset.target.astype(int)

    sample_size = min(40_000, len(X_full))
    if sample_size < len(X_full):
        X_sample, _, y_sample, _ = train_test_split(
            X_full,
            y_full,
            train_size=sample_size,
            stratify=y_full,
            random_state=RANDOM_STATE,
        )
    else:
        X_sample, y_sample = X_full, y_full

    class_counts = y_sample.value_counts().sort_index()
    class_dist = pd.DataFrame(
        {
            "class": class_counts.index,
            "count": class_counts.values,
            "percentage": safe_percentage(class_counts).values,
        }
    )
    save_dataframe(class_dist, out_dir / "class_distribution.csv")

    X_train, X_test, y_train, y_test = train_test_split(
        X_sample,
        y_sample,
        test_size=0.2,
        stratify=y_sample,
        random_state=RANDOM_STATE,
    )

    sampling_strategies = {
        "none": None,
        "smote": SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.1),
        "undersample": RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.1),
    }

    rows = []
    best = None
    best_curve = None
    model_factories = build_q2_models()

    for sampling_name, sampler in sampling_strategies.items():
        for model_name, base_estimator in model_factories.items():
            if sampler is None:
                estimator = clone(base_estimator)
            else:
                estimator = ImbPipeline(
                    [("sampler", sampler), ("model_pipeline", clone(base_estimator))]
                )
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_test)
            if hasattr(estimator, "predict_proba"):
                y_score = estimator.predict_proba(X_test)[:, 1]
            else:
                y_score = estimator.decision_function(X_test)

            metrics = {
                "sampling": sampling_name,
                "model": model_name,
                "precision": metric_float(precision_score(y_test, y_pred, zero_division=0)),
                "recall": metric_float(recall_score(y_test, y_pred, zero_division=0)),
                "f1": metric_float(f1_score(y_test, y_pred, zero_division=0)),
                "roc_auc": metric_float(roc_auc_score(y_test, y_score)),
            }
            rows.append(metrics)

            cm = confusion_matrix(y_test, y_pred)
            disp = ConfusionMatrixDisplay(cm)
            disp.plot(cmap="Blues", colorbar=False)
            plt.title(f"{model_name} - {sampling_name}")
            cm_path = out_dir / f"confusion_{sampling_name}_{model_name.lower().replace(' ', '_')}.png"
            save_plot(cm_path)

            if best is None or metrics["f1"] > best["f1"]:
                best = metrics
                fpr, tpr, _ = roc_curve(y_test, y_score)
                best_curve = (fpr, tpr, cm_path)

    results = pd.DataFrame(rows).sort_values(["f1", "roc_auc"], ascending=False)
    save_dataframe(results, out_dir / "classification_metrics.csv")

    plt.figure(figsize=(6, 5))
    plt.plot(best_curve[0], best_curve[1], label=f"{best['model']} ({best['sampling']})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve for Best Q2 Model")
    plt.legend()
    roc_path = out_dir / "best_roc_curve.png"
    save_plot(roc_path)

    summary = {
        "dataset_rows": int(len(X_sample)),
        "positive_rate_pct": metric_float(class_dist.loc[class_dist["class"] == 1, "percentage"].iloc[0]),
        "best_model": best["model"],
        "best_sampling": best["sampling"],
        "best_f1": metric_float(best["f1"]),
        "best_recall": metric_float(best["recall"]),
        "best_precision": metric_float(best["precision"]),
        "best_roc_auc": metric_float(best["roc_auc"]),
        "figures": {
            "roc_curve": relative_to_report(roc_path),
            "best_confusion_matrix": relative_to_report(best_curve[2]),
        },
        "metrics_file": relative_to_report(out_dir / "classification_metrics.csv"),
        "distribution_file": relative_to_report(out_dir / "class_distribution.csv"),
    }
    return {
        "summary": summary,
        "table": results,
        "distribution": class_dist,
    }


def run_question3() -> dict:
    out_dir = question_dir("question3")
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    X = mnist.data.astype(np.float32) / 255.0
    y = mnist.target.astype(int)

    sample_size = 4000
    X_subset, _, y_subset, _ = train_test_split(
        X,
        y,
        train_size=sample_size,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    pca_50 = PCA(n_components=50, random_state=RANDOM_STATE)
    X_pca_50 = pca_50.fit_transform(X_subset)
    pca_2 = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca_2 = pca_2.fit_transform(X_subset)

    tsne_30 = build_tsne(perplexity=30)
    X_tsne_30 = tsne_30.fit_transform(X_subset)

    tsne_50 = build_tsne(perplexity=50)
    X_tsne_50 = tsne_50.fit_transform(X_subset)

    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        x=X_pca_2[:, 0],
        y=X_pca_2[:, 1],
        hue=y_subset,
        palette="tab10",
        s=12,
        linewidth=0,
        legend=False,
    )
    plt.title("PCA 2D Projection")
    pca_fig = out_dir / "pca_2d.png"
    save_plot(pca_fig)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, embedding, title in zip(
        axes,
        [X_tsne_30, X_tsne_50],
        ["t-SNE (perplexity=30)", "t-SNE (perplexity=50)"],
    ):
        sns.scatterplot(
            x=embedding[:, 0],
            y=embedding[:, 1],
            hue=y_subset,
            palette="tab10",
            s=10,
            linewidth=0,
            legend=False,
            ax=ax,
        )
        ax.set_title(title)
    tsne_fig = out_dir / "tsne_2d.png"
    save_plot(tsne_fig)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    knn = KNeighborsClassifier(n_neighbors=5)
    cv_rows = []
    feature_sets = {
        "Original 784D": X_subset,
        "PCA 50D": X_pca_50,
        "t-SNE 2D": X_tsne_30,
    }
    for name, features in feature_sets.items():
        scores = cross_val_score(knn, features, y_subset, cv=cv, scoring="accuracy", n_jobs=-1)
        cv_rows.append(
            {
                "feature_space": name,
                "cv_accuracy_mean": metric_float(scores.mean()),
                "cv_accuracy_std": metric_float(scores.std()),
            }
        )

    explained_variance = pd.DataFrame(
        {
            "component": np.arange(1, 11),
            "explained_variance_ratio": pca_50.explained_variance_ratio_[:10],
        }
    )
    save_dataframe(explained_variance, out_dir / "explained_variance.csv")

    cv_results = pd.DataFrame(cv_rows).sort_values("cv_accuracy_mean", ascending=False)
    save_dataframe(cv_results, out_dir / "knn_cv_results.csv")

    summary = {
        "subset_rows": int(sample_size),
        "best_feature_space": cv_results.iloc[0]["feature_space"],
        "best_cv_accuracy": metric_float(cv_results.iloc[0]["cv_accuracy_mean"]),
        "pca_first10_cumulative": metric_float(explained_variance["explained_variance_ratio"].sum()),
        "figures": {
            "pca_2d": relative_to_report(pca_fig),
            "tsne_2d": relative_to_report(tsne_fig),
        },
        "variance_file": relative_to_report(out_dir / "explained_variance.csv"),
        "metrics_file": relative_to_report(out_dir / "knn_cv_results.csv"),
    }
    return {
        "summary": summary,
        "variance_table": explained_variance,
        "cv_table": cv_results,
    }




def build_tsne(perplexity: int) -> TSNE:
    tsne_kwargs = {
        "n_components": 2,
        "perplexity": perplexity,
        "init": "pca",
        "learning_rate": "auto",
        "random_state": RANDOM_STATE,
    }
    if "max_iter" in inspect.signature(TSNE).parameters:
        tsne_kwargs["max_iter"] = 1000
    else:
        tsne_kwargs["n_iter"] = 1000
    return TSNE(**tsne_kwargs)

def clustering_scores(X: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    mask = labels != -1
    if mask.sum() == 0 or len(np.unique(labels[mask])) < 2:
        return float("nan"), float("nan")
    X_eval = X[mask]
    labels_eval = labels[mask]
    return (
        metric_float(silhouette_score(X_eval, labels_eval)),
        metric_float(davies_bouldin_score(X_eval, labels_eval)),
    )


def run_question4() -> dict:
    out_dir = question_dir("question4")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
    df = pd.read_csv(url)
    features = df.columns.tolist()
    X = df[features].to_numpy(dtype=float)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ks = list(range(2, 11))
    k_rows = []
    best_k = None
    best_sil = -np.inf
    for k in ks:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(X_scaled)
        sil, dbi = clustering_scores(X_scaled, labels)
        k_rows.append({"k": k, "inertia": metric_float(model.inertia_), "silhouette": sil, "dbi": dbi})
        if not math.isnan(sil) and sil > best_sil:
            best_sil = sil
            best_k = k

    k_table = pd.DataFrame(k_rows)
    save_dataframe(k_table, out_dir / "kmeans_search.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(k_table["k"], k_table["inertia"], marker="o")
    axes[0].set_title("Elbow Curve")
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Inertia")
    axes[1].plot(k_table["k"], k_table["silhouette"], marker="o", color="#0f766e")
    axes[1].set_title("Silhouette by k")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Silhouette")
    elbow_path = out_dir / "kmeans_selection.png"
    save_plot(elbow_path)

    kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
    kmeans_labels = kmeans.fit_predict(X_scaled)

    agglomerative = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
    agg_labels = agglomerative.fit_predict(X_scaled)

    neighbors = NearestNeighbors(n_neighbors=5)
    neighbors.fit(X_scaled)
    distances, _ = neighbors.kneighbors(X_scaled)
    k_distances = np.sort(distances[:, -1])

    plt.figure(figsize=(7, 4.5))
    plt.plot(k_distances)
    plt.title("k-distance Graph (k=5)")
    plt.xlabel("Points sorted by distance")
    plt.ylabel("5th NN distance")
    kdist_path = out_dir / "dbscan_k_distance.png"
    save_plot(kdist_path)

    eps_candidates = np.quantile(k_distances, [0.8, 0.85, 0.9, 0.92, 0.95, 0.97]).tolist()
    best_dbscan = None
    best_dbscan_score = -np.inf
    for eps in eps_candidates:
        dbscan = DBSCAN(eps=float(eps), min_samples=5)
        labels = dbscan.fit_predict(X_scaled)
        sil, dbi = clustering_scores(X_scaled, labels)
        if not math.isnan(sil) and sil > best_dbscan_score:
            best_dbscan_score = sil
            best_dbscan = {"eps": float(eps), "labels": labels, "silhouette": sil, "dbi": dbi}

    if best_dbscan is None:
        fallback = DBSCAN(eps=float(np.quantile(k_distances, 0.95)), min_samples=5)
        labels = fallback.fit_predict(X_scaled)
        sil, dbi = clustering_scores(X_scaled, labels)
        best_dbscan = {
            "eps": float(np.quantile(k_distances, 0.95)),
            "labels": labels,
            "silhouette": sil,
            "dbi": dbi,
        }

    metrics_rows = []
    cluster_labels = {
        "K-Means": kmeans_labels,
        "Agglomerative": agg_labels,
        "DBSCAN": best_dbscan["labels"],
    }
    for name, labels in cluster_labels.items():
        sil, dbi = clustering_scores(X_scaled, labels)
        metrics_rows.append(
            {
                "algorithm": name,
                "clusters_found": int(len(np.unique(labels[labels != -1]))),
                "noise_points": int((labels == -1).sum()),
                "silhouette": sil,
                "davies_bouldin": dbi,
            }
        )

    metrics_table = pd.DataFrame(metrics_rows).sort_values("silhouette", ascending=False)
    save_dataframe(metrics_table, out_dir / "clustering_metrics.csv")

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_2d = pca.fit_transform(X_scaled)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    for ax, (name, labels) in zip(axes, cluster_labels.items()):
        sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette="tab10", ax=ax, s=25, legend=False)
        ax.set_title(name)
    cluster_fig = out_dir / "cluster_projection.png"
    save_plot(cluster_fig)

    interpretation = (
        pd.DataFrame(df)
        .assign(kmeans_cluster=kmeans_labels)
        .groupby("kmeans_cluster")
        .mean(numeric_only=True)
        .round(2)
        .reset_index()
    )
    save_dataframe(interpretation, out_dir / "kmeans_cluster_profiles.csv")

    summary = {
        "dataset_rows": int(len(df)),
        "best_k": int(best_k),
        "best_algorithm": metrics_table.iloc[0]["algorithm"],
        "best_silhouette": metric_float(metrics_table.iloc[0]["silhouette"]),
        "dbscan_eps": metric_float(best_dbscan["eps"]),
        "figures": {
            "kmeans_selection": relative_to_report(elbow_path),
            "dbscan_k_distance": relative_to_report(kdist_path),
            "cluster_projection": relative_to_report(cluster_fig),
        },
        "metrics_file": relative_to_report(out_dir / "clustering_metrics.csv"),
        "profile_file": relative_to_report(out_dir / "kmeans_cluster_profiles.csv"),
    }
    return {
        "summary": summary,
        "search_table": k_table,
        "metrics_table": metrics_table,
        "profile_table": interpretation,
    }


class IndexedDataset(Dataset):
    def __init__(self, dataset: Dataset):
        self.dataset = dataset

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int):
        image, label = self.dataset[idx]
        return image, label, idx


class MLPNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)


class CNNNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device, criterion: nn.Module):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_images = []
    all_indices = []
    with torch.no_grad():
        for images, labels, indices in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_images.extend(images.cpu().numpy())
            all_indices.extend(indices.numpy().tolist())
    mean_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    return {
        "loss": mean_loss,
        "accuracy": accuracy,
        "predictions": np.array(all_preds),
        "labels": np.array(all_labels),
        "images": np.array(all_images),
        "indices": np.array(all_indices),
    }


def train_torch_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
) -> dict:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    patience = 1
    max_epochs = 3
    best_state = None
    best_val_loss = float("inf")
    history = []
    patience_counter = 0

    model.to(device)

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        all_train_preds = []
        all_train_labels = []
        for images, labels, _ in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            all_train_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
            all_train_labels.extend(labels.detach().cpu().numpy().tolist())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        val_metrics = evaluate_model(model, val_loader, device, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": metric_float(train_loss),
                "train_accuracy": metric_float(train_acc),
                "val_loss": metric_float(val_metrics["loss"]),
                "val_accuracy": metric_float(val_metrics["accuracy"]),
            }
        )
        print(f"{model_name} epoch {epoch}: train_acc={train_acc:.4f}, val_acc={val_metrics['accuracy']:.4f}, val_loss={val_metrics['loss']:.4f}")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader, device, criterion)
    macro_f1 = f1_score(test_metrics["labels"], test_metrics["predictions"], average="macro")
    return {
        "model_name": model_name,
        "history": pd.DataFrame(history),
        "test_accuracy": metric_float(test_metrics["accuracy"]),
        "macro_f1": metric_float(macro_f1),
        "predictions": test_metrics["predictions"],
        "labels": test_metrics["labels"],
        "images": test_metrics["images"],
    }


def plot_training_curves(history: pd.DataFrame, title: str, path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(history["epoch"], history["train_loss"], label="Train")
    axes[0].plot(history["epoch"], history["val_loss"], label="Validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history["epoch"], history["train_accuracy"], label="Train")
    axes[1].plot(history["epoch"], history["val_accuracy"], label="Validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    save_plot(path)


def plot_misclassifications(images: np.ndarray, labels: np.ndarray, preds: np.ndarray, path: Path) -> None:
    wrong = np.where(labels != preds)[0][:5]
    fig, axes = plt.subplots(1, 5, figsize=(12, 3))
    for ax, idx in zip(axes, wrong):
        ax.imshow(images[idx].squeeze(), cmap="gray")
        ax.set_title(f"T:{labels[idx]} P:{preds[idx]}")
        ax.axis("off")
    save_plot(path)


def run_question5() -> dict:
    out_dir = question_dir("question5")
    data_root = DATA_DIR / "fashion_mnist"
    transform = transforms.ToTensor()
    train_base = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
    test_base = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)

    generator = torch.Generator().manual_seed(RANDOM_STATE)
    working_train_size = min(12000, len(train_base))
    remaining_train_size = len(train_base) - working_train_size
    working_train, _ = random_split(train_base, [working_train_size, remaining_train_size], generator=generator)

    train_size = int(0.8 * len(working_train))
    val_size = len(working_train) - train_size
    train_split, val_split = random_split(working_train, [train_size, val_size], generator=generator)

    working_test_size = min(2000, len(test_base))
    remaining_test_size = len(test_base) - working_test_size
    test_split, _ = random_split(test_base, [working_test_size, remaining_test_size], generator=generator)

    train_data = IndexedDataset(train_split)
    val_data = IndexedDataset(val_split)
    test_data = IndexedDataset(test_split)

    print(f"Q5 working sizes -> train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}")

    batch_size = 512
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    device = torch.device("cpu")
    results = []
    histories = {}
    figures = {}

    for name, model in [("MLP", MLPNet()), ("CNN", CNNNet())]:
        print(f"Starting {name} on {device}...")
        outcome = train_torch_model(model, name, train_loader, val_loader, test_loader, device)
        history = outcome["history"]
        histories[name] = history

        curve_path = out_dir / f"{name.lower()}_training_curves.png"
        plot_training_curves(history, name, curve_path)

        cm = confusion_matrix(outcome["labels"], outcome["predictions"])
        disp = ConfusionMatrixDisplay(cm)
        disp.plot(cmap="Blues", colorbar=False)
        plt.title(f"{name} Confusion Matrix")
        cm_path = out_dir / f"{name.lower()}_confusion_matrix.png"
        save_plot(cm_path)

        miscls_path = out_dir / f"{name.lower()}_misclassified_examples.png"
        plot_misclassifications(
            outcome["images"], outcome["labels"], outcome["predictions"], miscls_path
        )

        history.to_csv(out_dir / f"{name.lower()}_history.csv", index=False)
        results.append(
            {
                "model": name,
                "accuracy": outcome["test_accuracy"],
                "macro_f1": outcome["macro_f1"],
                "epochs_trained": int(history["epoch"].iloc[-1]),
            }
        )
        figures[name] = {
            "curves": relative_to_report(curve_path),
            "confusion_matrix": relative_to_report(cm_path),
            "misclassified_examples": relative_to_report(miscls_path),
        }

    results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
    save_dataframe(results_df, out_dir / "neural_network_metrics.csv")
    best_row = results_df.iloc[0]

    summary = {
        "device": str(device),
        "best_model": best_row["model"],
        "best_accuracy": metric_float(best_row["accuracy"]),
        "best_macro_f1": metric_float(best_row["macro_f1"]),
        "metrics_file": relative_to_report(out_dir / "neural_network_metrics.csv"),
        "figures": figures,
    }
    return {
        "summary": summary,
        "results_table": results_df,
    }

## Initialize Workspace

In [3]:
seed_everything()
ensure_dirs()
print(ROOT)

/Users/alperenertan/Documents/New project 2


## Question 1

California Housing regression comparison with baseline, scaled, and polynomial feature settings.

In [4]:
q1 = run_question1()
q1["table"]

,experiment,model,RMSE,MAE,R2
0,baseline,Linear Regression,0.745581,0.533200,0.575788
1,baseline,Ridge Regression,0.745010,0.533243,0.576437
2,baseline,Random Forest,0.503960,0.326812,0.806186
3,scaled,Linear Regression,0.745581,0.533200,0.575788
4,scaled,Ridge Regression,0.745538,0.533188,0.575837
5,scaled,Random Forest,0.503802,0.326753,0.806307
6,polynomial_degree_2,Linear Regression,0.681397,0.467001,0.645682
7,polynomial_degree_2,Ridge Regression,0.747766,0.529114,0.573298
8,polynomial_degree_2,Random Forest,0.512900,0.334738,0.799249


## Question 2

Imbalanced fraud classification with no resampling, SMOTE, and random undersampling.

In [5]:
q2 = run_question2()
q2["distribution"], q2["table"]

(   class  count  percentage
 0      0  39931     99.8275
 1      1     69      0.1725,
       sampling                model  precision    recall        f1   roc_auc
 4        smote        Random Forest   1.000000  0.785714  0.880000  0.997138
 5        smote              XGBoost   1.000000  0.785714  0.880000  0.992138
 2         none              XGBoost   1.000000  0.714286  0.833333  0.971870
 8  undersample              XGBoost   0.785714  0.785714  0.785714  0.972049
 1         none        Random Forest   1.000000  0.642857  0.782609  0.925369
 0         none  Logistic Regression   0.800000  0.571429  0.666667  0.989884
 7  undersample        Random Forest   0.500000  0.785714  0.611111  0.966151
 3        smote  Logistic Regression   0.379310  0.785714  0.511628  0.978489
 6  undersample  Logistic Regression   0.312500  0.714286  0.434783  0.985090)

## Question 3

PCA and t-SNE on MNIST, followed by k-NN comparisons.

In [6]:
q3 = run_question3()
q3["variance_table"], q3["cv_table"]

(   component  explained_variance_ratio
 0          1                  0.097886
 1          2                  0.072981
 2          3                  0.061771
 3          4                  0.054759
 4          5                  0.050018
 5          6                  0.043268
 6          7                  0.032007
 7          8                  0.029337
 8          9                  0.027212
 9         10                  0.024283,
    feature_space  cv_accuracy_mean  cv_accuracy_std
 1        PCA 50D            0.9425         0.007984
 0  Original 784D            0.9300         0.007202
 2       t-SNE 2D            0.9275         0.012222)

## Question 4

Clustering comparison on the Wholesale Customers dataset.

In [7]:
q4 = run_question4()
q4["search_table"], q4["metrics_table"], q4["profile_table"]

(    k      inertia  silhouette       dbi
 0   2  2599.385559    0.373236  1.260319
 1   3  2149.283956    0.356769  1.173637
 2   4  1847.399351    0.368314  1.145391
 3   5  1541.530265    0.350201  1.127769
 4   6  1313.956879    0.352584  0.958810
 5   7  1173.728596    0.363482  0.849515
 6   8  1050.969742    0.363137  0.935440
 7   9   975.032127    0.353625  1.016683
 8  10   913.565211    0.357619  0.953265,
        algorithm  clusters_found  noise_points  silhouette  davies_bouldin
 2         DBSCAN               2            48    0.415998        1.044902
 0        K-Means               2             0    0.373236        1.260319
 1  Agglomerative               2             0    0.368021        1.284974,
    kmeans_cluster  Channel  Region     Fresh      Milk   Grocery   Frozen  \
 0               0     1.99    2.63   8508.32  11337.74  16914.15  1722.54   
 1               1     1.02    2.50  13562.50   3317.18   3941.57  3675.61   
 
    Detergents_Paper  Delicassen  
 0 

## Question 5

MLP and CNN comparison on Fashion-MNIST.

In [ ]:
q5 = run_question5()
q5["results_table"]

## Summary

In [ ]:
summary = {
    "question1": q1["summary"],
    "question2": q2["summary"],
    "question3": q3["summary"],
    "question4": q4["summary"],
    "question5": q5["summary"],
}
summary